# 倾角引导的多道稀疏反射系数反演（深度域）

本 notebook 直接处理真实深度域地震。目标是在约 20–50 m 的尺度上，把宽波瓣中横向相干的弱反射从叠加波形中分离出来。

- raw seismic 决定振幅和重褶积闭合；
- dynamic-gain balanced seismic 只调整稀疏先验的空间权重；
- 邻道先局部倾角对齐，再通过 group sparsity 共享事件位置；
- 输出是确定性的反射系数与相对 log-AI 增量，不把地震波瓣过零点直接当作界面。

本轮使用井曲线只做结果对照。真正的反演目标函数不读取井曲线。

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal
from scipy.ndimage import gaussian_filter1d

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if not (repo_root / 'src').is_dir():
    raise RuntimeError('Could not locate repository root.')
sys.path.insert(0, str(repo_root / 'src'))

from cup.seismic.survey import open_survey

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.grid': True, 'grid.alpha': 0.18})

SMOKE = os.environ.get('SPARSE_REFLECTIVITY_SMOKE', '0') == '1'
RAW_SEISMIC = repo_root / 'data' / 'raw' / 'mero_84_coord_extend'
BALANCED_SEISMIC = repo_root / 'experiments' / 'dynamic_gain_balancing' / 'results' / '20260811_nw11_depth' / 'dynamic_gain_balanced_seismic_nw11_depth.segy'
WAVELET_FILE = repo_root / 'scripts' / 'output' / 'vertical_well_auto_tie_depth_20260719_172336' / 'wavelet_201ms_NW11.csv'
METRICS_FILE = repo_root / 'scripts' / 'output' / 'wavelet_batch_synthetic_depth_20260719_172510' / 'wavelet_batch_metrics.csv'
FROZEN_WELLS = repo_root / 'note' / 'summary' / 'final_audit' / '20260810_well_prior_texture_and_decoder_failure' / 'results' / 'well_residual_decomposition' / '20260810_body_scale_decomposition' / 'wells'
RUN_ID = '20260811_dip_guided_sparse_reflectivity_smoke' if SMOKE else '20260811_dip_guided_sparse_reflectivity'
OUTPUT_DIR = repo_root / 'experiments' / 'sparse_reflectivity' / 'results' / RUN_ID
FIGURE_DIR = OUTPUT_DIR / 'figures'
WELL_DIR = OUTPUT_DIR / 'wells'
for path in (OUTPUT_DIR, FIGURE_DIR, WELL_DIR):
    path.mkdir(parents=True, exist_ok=True)

SEGY_OPTIONS = {'iline': 5, 'xline': 21, 'istep': 1, 'xstep': 4}
TRUSTED_WELLS = ('2-ANP-2A-RJS', 'L1-NW1', 'L5-NW5', 'L9-NW4A', 'NW11', 'NW8')
SELECTED_WELLS = ('NW11',) if SMOKE else TRUSTED_WELLS
PATCH_RADIUS = 4 if SMOKE else 10
ZONE_MARGIN_M = 100.0
ALIGNMENT_WINDOW_M = 100.0
MAX_LOCAL_SHIFT_M = 20.0
TARGET_MIN_EVENT_SPACING_M = 20.0
GROUP_LAMBDA_FRACTION = 0.055
INDIVIDUAL_LAMBDA_FRACTION = 0.008
LATERAL_SMOOTHNESS_FRACTION = 0.015
MAX_ITERATIONS = 180 if SMOKE else 500
CONVERGENCE_TOLERANCE = 2e-5

for required in (RAW_SEISMIC, BALANCED_SEISMIC, WAVELET_FILE, METRICS_FILE, FROZEN_WELLS):
    if not required.exists():
        raise FileNotFoundError(required)

print('Smoke:', SMOKE)
print('Wells:', SELECTED_WELLS)
print('Output:', OUTPUT_DIR)

## 1. 局部剖面与倾角对齐

每口井取 xline 方向的局部 patch。`xline_step=4` 只用于线号寻址；横向图件使用 survey 几何给出的实际米制道间距。每条邻道在滑动窗口内与中心道相关，得到随深度平滑变化的局部 shift。

In [ ]:
def finite_corr(left, right):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)
    valid = np.isfinite(left) & np.isfinite(right)
    if int(valid.sum()) < 3 or np.std(left[valid]) <= 1e-12 or np.std(right[valid]) <= 1e-12:
        return np.nan
    return float(np.corrcoef(left[valid], right[valid])[0, 1])


def regular_sample_step(axis):
    axis = np.asarray(axis, dtype=np.float64)
    steps = np.diff(axis)
    step = float(np.median(steps))
    if step <= 0.0 or not np.allclose(steps, step, rtol=1e-6, atol=1e-6):
        raise ValueError('Expected a regular increasing sample axis.')
    return step


def extract_xline_patch(survey, well_x, well_y, sample_start, sample_end, radius):
    i_float, j_float = survey.line_geometry.coord_to_index(float(well_x), float(well_y))
    i_center = int(round(i_float))
    j_center = int(round(j_float))
    offsets = np.arange(-int(radius), int(radius) + 1, dtype=int)
    indices = [(i_center, j_center + int(offset)) for offset in offsets]
    traces = survey.read_traces_at_indices(indices, sample_start, sample_end, domain='depth')
    ordered = [traces[index] for index in indices]
    axis = np.asarray(ordered[0].basis, dtype=np.float64)
    values = np.stack([np.asarray(trace.values, dtype=np.float64) for trace in ordered])
    for trace in ordered[1:]:
        if not np.array_equal(axis, np.asarray(trace.basis, dtype=np.float64)):
            raise ValueError('Patch traces do not share one sample axis.')
    exact_center = survey.read_trace_at_xy(well_x, well_y, sample_start, sample_end, domain='depth')
    if not np.array_equal(axis, np.asarray(exact_center.basis, dtype=np.float64)):
        raise ValueError('Bilinear center trace does not share the patch sample axis.')
    values[int(radius)] = np.asarray(exact_center.values, dtype=np.float64)
    spacing = survey.line_geometry.bin_spacing_m()['xline_spacing_m']
    lateral_m = offsets.astype(np.float64) * float(spacing)
    return axis, lateral_m, values, (i_center, j_center)


def local_alignment_shifts(center, trace, half_window, max_shift):
    center = np.asarray(center, dtype=np.float64)
    trace = np.asarray(trace, dtype=np.float64)
    n = center.size
    shifts = np.zeros(n, dtype=np.float64)
    for sample in range(n):
        start = max(0, sample - half_window)
        stop = min(n, sample + half_window + 1)
        reference = center[start:stop]
        best_score = -np.inf
        best_shift = 0
        for lag in range(-max_shift, max_shift + 1):
            other_start = start + lag
            other_stop = stop + lag
            if other_start < 0 or other_stop > n:
                continue
            score = finite_corr(reference, trace[other_start:other_stop])
            if np.isfinite(score) and score > best_score:
                best_score = score
                best_shift = lag
        shifts[sample] = float(best_shift)
    return gaussian_filter1d(shifts, sigma=max(1.0, half_window / 4.0), mode='nearest')


def align_patch(reference_patch, moving_patch, dz_m):
    reference_patch = np.asarray(reference_patch, dtype=np.float64)
    moving_patch = np.asarray(moving_patch, dtype=np.float64)
    center_index = reference_patch.shape[0] // 2
    half_window = max(2, int(round(0.5 * ALIGNMENT_WINDOW_M / dz_m)))
    max_shift = max(1, int(round(MAX_LOCAL_SHIFT_M / dz_m)))
    sample_index = np.arange(reference_patch.shape[1], dtype=np.float64)
    shifts = np.zeros(reference_patch.shape, dtype=np.float64)
    aligned = np.empty_like(moving_patch)
    for trace_index in range(reference_patch.shape[0]):
        if trace_index == center_index:
            aligned[trace_index] = moving_patch[trace_index]
            continue
        local_shift = local_alignment_shifts(
            reference_patch[center_index], reference_patch[trace_index], half_window, max_shift
        )
        shifts[trace_index] = local_shift
        aligned[trace_index] = np.interp(
            sample_index + local_shift, sample_index, moving_patch[trace_index], left=np.nan, right=np.nan
        )
    return aligned, shifts


def unalign_patch(aligned, shifts):
    aligned = np.asarray(aligned, dtype=np.float64)
    sample_index = np.arange(aligned.shape[1], dtype=np.float64)
    output = np.full_like(aligned, np.nan)
    for trace_index in range(aligned.shape[0]):
        mapped = sample_index + shifts[trace_index]
        mapped = np.maximum.accumulate(mapped + 1e-6 * sample_index)
        output[trace_index] = np.interp(sample_index, mapped, aligned[trace_index], left=np.nan, right=np.nan)
    return output

## 2. 深度域子波与联合稀疏反褶积

目标函数由 raw seismic 的重褶积误差、逐道稀疏项、跨道 group sparsity 和很弱的横向振幅平滑组成。balanced seismic 经过匹配滤波后只生成深度相关的 penalty weight。

In [ ]:
def depth_sampled_wavelet(time_s, amplitude, dz_m, velocity_mps):
    time_s = np.asarray(time_s, dtype=np.float64)
    amplitude = np.asarray(amplitude, dtype=np.float64)
    max_depth_lag = 0.5 * float(velocity_mps) * float(np.max(np.abs(time_s)))
    half_samples = int(np.ceil(max_depth_lag / dz_m))
    depth_lags = np.arange(-half_samples, half_samples + 1, dtype=np.float64) * dz_m
    equivalent_time = 2.0 * depth_lags / float(velocity_mps)
    wavelet = np.interp(equivalent_time, time_s, amplitude, left=0.0, right=0.0)
    wavelet -= np.mean(wavelet)
    return depth_lags, wavelet


def convolve_rows(values, wavelet):
    return signal.fftconvolve(np.asarray(values), np.asarray(wavelet)[None, :], mode='same', axes=-1)


def adjoint_rows(values, wavelet):
    return signal.fftconvolve(np.asarray(values), np.asarray(wavelet)[None, ::-1], mode='same', axes=-1)


def lateral_laplacian(values):
    output = np.zeros_like(values)
    output[0] = values[0] - values[1]
    output[-1] = values[-1] - values[-2]
    output[1:-1] = 2.0 * values[1:-1] - values[:-2] - values[2:]
    return output


def balanced_penalty_weights(raw, balanced, wavelet):
    raw_match = np.median(np.abs(adjoint_rows(raw, wavelet)), axis=0)
    balanced_match = np.median(np.abs(adjoint_rows(balanced, wavelet)), axis=0)
    evidence = np.maximum(raw_match, balanced_match)
    scale = float(np.percentile(evidence[np.isfinite(evidence)], 95.0))
    normalized = np.clip(evidence / max(scale, 1e-12), 0.0, 1.5)
    return np.clip(1.10 - 0.65 * normalized, 0.35, 1.10)


def sparse_group_prox(values, individual_threshold, group_threshold):
    sparse = np.sign(values) * np.maximum(np.abs(values) - individual_threshold, 0.0)
    norms = np.sqrt(np.sum(sparse * sparse, axis=0))
    factors = np.maximum(1.0 - group_threshold / np.maximum(norms, 1e-12), 0.0)
    return sparse * factors[None, :]


def solve_multitrace_reflectivity(raw, balanced, wavelet):
    raw = np.nan_to_num(np.asarray(raw, dtype=np.float64))
    balanced = np.nan_to_num(np.asarray(balanced, dtype=np.float64))
    matched = adjoint_rows(raw, wavelet)
    group_lambda_max = float(np.max(np.sqrt(np.sum(matched * matched, axis=0))))
    individual_lambda_max = float(np.max(np.abs(matched)))
    group_lambda = GROUP_LAMBDA_FRACTION * group_lambda_max
    individual_lambda = INDIVIDUAL_LAMBDA_FRACTION * individual_lambda_max
    penalty_weight = balanced_penalty_weights(raw, balanced, wavelet)

    fft_size = int(2 ** np.ceil(np.log2(raw.shape[1] + wavelet.size - 1)))
    convolution_lipschitz = float(np.max(np.abs(np.fft.rfft(wavelet, n=fft_size)) ** 2))
    lateral_weight = LATERAL_SMOOTHNESS_FRACTION * convolution_lipschitz
    step = 0.95 / max(convolution_lipschitz + 4.0 * lateral_weight, 1e-12)

    reflectivity = np.zeros_like(raw)
    extrapolated = reflectivity.copy()
    momentum = 1.0
    history = []
    for iteration in range(1, MAX_ITERATIONS + 1):
        residual = convolve_rows(extrapolated, wavelet) - raw
        gradient = adjoint_rows(residual, wavelet) + lateral_weight * lateral_laplacian(extrapolated)
        proposal = extrapolated - step * gradient
        updated = sparse_group_prox(
            proposal,
            step * individual_lambda,
            step * group_lambda * penalty_weight,
        )
        relative_change = float(np.linalg.norm(updated - reflectivity) / max(np.linalg.norm(reflectivity), 1e-12))
        next_momentum = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * momentum * momentum))
        extrapolated = updated + ((momentum - 1.0) / next_momentum) * (updated - reflectivity)
        reflectivity = updated
        momentum = next_momentum
        if iteration == 1 or iteration % 25 == 0 or relative_change < CONVERGENCE_TOLERANCE:
            data_misfit = 0.5 * float(np.mean(residual * residual))
            history.append({'iteration': iteration, 'data_misfit': data_misfit, 'relative_change': relative_change})
        if iteration >= 50 and relative_change < CONVERGENCE_TOLERANCE:
            break
    reconstruction = convolve_rows(reflectivity, wavelet)
    return {
        'reflectivity': reflectivity,
        'reconstruction': reconstruction,
        'penalty_weight': penalty_weight,
        'history': pd.DataFrame(history),
        'iterations': int(history[-1]['iteration']),
        'final_relative_change': float(history[-1]['relative_change']),
        'converged': bool(history[-1]['relative_change'] < CONVERGENCE_TOLERANCE),
        'group_lambda': group_lambda,
        'individual_lambda': individual_lambda,
    }

## 3. 固定 NW11 子波尺度并运行井位局部剖面

子波相位和尺度由 NW11 标定结果固定。其他井不再各自自由拟合 gain。每个 patch 使用中心道的原始 RMS 做共同标准化，从而保留邻道之间的相对振幅。

In [ ]:
metrics = pd.read_csv(METRICS_FILE).set_index('well_name')
wavelet_frame = pd.read_csv(WAVELET_FILE)
wavelet_time_s = wavelet_frame['time_s'].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet_frame['amplitude'].to_numpy(dtype=np.float64)
nw11_forward_scale = float(metrics.loc['NW11', 'scale'])

raw_survey = open_survey(RAW_SEISMIC, seismic_type='segy', segy_options=SEGY_OPTIONS)
balanced_survey = open_survey(BALANCED_SEISMIC, seismic_type='segy', segy_options=SEGY_OPTIONS)
raw_geometry = raw_survey.describe_geometry(domain='depth')
balanced_geometry = balanced_survey.describe_geometry(domain='depth')
for key in ('inline_min', 'inline_max', 'inline_step', 'xline_min', 'xline_max', 'xline_step', 'sample_min', 'sample_max', 'sample_step'):
    if raw_geometry[key] != balanced_geometry[key]:
        raise ValueError(f'Raw/balanced geometry differs at {key}.')

results = {}
metric_rows = []
for well_name in SELECTED_WELLS:
    with np.load(FROZEN_WELLS / f'{well_name}.npz', allow_pickle=False) as artifact:
        highres_depth = artifact['tvdss_m'].astype(np.float64)
        highres_log_ai = artifact['well_log_ai'].astype(np.float64)
        highres_valid = artifact['well_valid'].astype(bool)
        horizon_depths = artifact['horizon_tvdss_m'].astype(np.float64)
    zone_top = float(np.min(horizon_depths))
    zone_bottom = float(np.max(horizon_depths))
    sample_start = zone_top - ZONE_MARGIN_M
    sample_end = zone_bottom + ZONE_MARGIN_M
    row = metrics.loc[well_name]
    raw_axis, lateral_m, raw_patch, center_index = extract_xline_patch(
        raw_survey, float(row['well_x']), float(row['well_y']), sample_start, sample_end, PATCH_RADIUS
    )
    balanced_axis, balanced_lateral_m, balanced_patch, balanced_center = extract_xline_patch(
        balanced_survey, float(row['well_x']), float(row['well_y']), sample_start, sample_end, PATCH_RADIUS
    )
    if not np.array_equal(raw_axis, balanced_axis) or not np.array_equal(lateral_m, balanced_lateral_m):
        raise ValueError(f'{well_name}: raw/balanced patch axes differ.')
    dz_m = regular_sample_step(raw_axis)
    center_row = raw_patch.shape[0] // 2
    raw_means = np.nanmean(raw_patch, axis=1, keepdims=True)
    balanced_means = np.nanmean(balanced_patch, axis=1, keepdims=True)
    common_scale = float(np.nanstd(raw_patch[center_row]))
    if not np.isfinite(common_scale) or common_scale <= 0.0:
        raise ValueError(f'{well_name}: center trace has invalid amplitude scale.')
    raw_normalized = (raw_patch - raw_means) / common_scale
    balanced_normalized = (balanced_patch - balanced_means) / common_scale
    aligned_raw, shifts = align_patch(balanced_normalized, raw_normalized, dz_m)
    aligned_balanced, balanced_shifts = align_patch(balanced_normalized, balanced_normalized, dz_m)
    if not np.allclose(shifts, balanced_shifts, atol=1e-9):
        raise ValueError(f'{well_name}: raw and balanced alignment shifts diverged.')

    _, local_wavelet = depth_sampled_wavelet(
        wavelet_time_s, wavelet_amplitude, dz_m, float(row['median_vp_mps'])
    )
    local_wavelet = local_wavelet * nw11_forward_scale
    solution = solve_multitrace_reflectivity(aligned_raw, aligned_balanced, local_wavelet)
    reflectivity = unalign_patch(solution['reflectivity'], shifts)
    reconstruction = unalign_patch(solution['reconstruction'], shifts)
    residual = raw_normalized - reconstruction

    center_r = np.nan_to_num(reflectivity[center_row])
    clipped_r = np.clip(center_r, -0.25, 0.25)
    relative_log_ai = np.zeros_like(clipped_r)
    relative_log_ai[1:] = np.cumsum(2.0 * np.arctanh(clipped_r[:-1]))
    relative_log_ai -= np.linspace(relative_log_ai[0], relative_log_ai[-1], relative_log_ai.size)

    truth_log_ai = np.full(raw_axis.shape, np.nan, dtype=np.float64)
    valid_truth = highres_valid & np.isfinite(highres_depth) & np.isfinite(highres_log_ai)
    if np.count_nonzero(valid_truth) >= 2:
        truth_log_ai = np.interp(raw_axis, highres_depth[valid_truth], highres_log_ai[valid_truth], left=np.nan, right=np.nan)
    truth_reflectivity = np.full(raw_axis.shape, np.nan, dtype=np.float64)
    finite_pairs = np.isfinite(truth_log_ai[:-1]) & np.isfinite(truth_log_ai[1:])
    truth_reflectivity[:-1][finite_pairs] = np.tanh(0.5 * np.diff(truth_log_ai)[finite_pairs])

    zone = (raw_axis >= zone_top) & (raw_axis <= zone_bottom)
    fit_support = zone & np.isfinite(reconstruction[center_row])
    reconstruction_corr = finite_corr(raw_normalized[center_row, fit_support], reconstruction[center_row, fit_support])
    reconstruction_rmse = float(np.sqrt(np.mean(residual[center_row, fit_support] ** 2)))
    min_distance = max(1, int(round(TARGET_MIN_EVENT_SPACING_M / dz_m)))
    zone_indices = np.flatnonzero(zone)
    local_abs = np.abs(center_r[zone])
    peak_threshold = 0.20 * float(np.percentile(local_abs, 95.0)) if local_abs.size else np.inf
    local_peaks, properties = signal.find_peaks(local_abs, height=peak_threshold, distance=min_distance)
    event_depths = raw_axis[zone_indices[local_peaks]] if local_peaks.size else np.empty(0)
    separations = np.diff(event_depths)

    result = {
        'well_name': well_name, 'axis_m': raw_axis, 'lateral_m': lateral_m, 'raw': raw_normalized,
        'balanced': balanced_normalized, 'aligned_raw': aligned_raw, 'reflectivity': reflectivity,
        'reconstruction': reconstruction, 'residual': residual, 'shifts_samples': shifts,
        'penalty_weight': solution['penalty_weight'], 'relative_log_ai': relative_log_ai,
        'truth_log_ai': truth_log_ai, 'truth_reflectivity': truth_reflectivity,
        'zone_top_m': zone_top, 'zone_bottom_m': zone_bottom, 'event_depths_m': event_depths,
        'wavelet': local_wavelet, 'history': solution['history'],
    }
    results[well_name] = result
    metric_rows.append({
        'well_name': well_name, 'trace_count': raw_patch.shape[0], 'sample_interval_m': dz_m,
        'xline_spacing_m': float(np.median(np.diff(lateral_m))), 'iterations': solution['iterations'],
        'converged': solution['converged'], 'final_relative_change': solution['final_relative_change'],
        'reconstruction_corr': reconstruction_corr, 'reconstruction_rmse_normalized': reconstruction_rmse,
        'event_count_20m_separated': int(event_depths.size),
        'event_separation_p50_m': float(np.median(separations)) if separations.size else np.nan,
        'event_separation_min_m': float(np.min(separations)) if separations.size else np.nan,
        'max_abs_local_shift_m': float(np.nanmax(np.abs(shifts)) * dz_m),
        'group_lambda': solution['group_lambda'], 'individual_lambda': solution['individual_lambda'],
    })
    np.savez_compressed(
        WELL_DIR / f'{well_name}.npz', axis_m=raw_axis, lateral_m=lateral_m, raw=raw_normalized,
        balanced=balanced_normalized, reflectivity=reflectivity, reconstruction=reconstruction,
        residual=residual, shifts_samples=shifts, penalty_weight=solution['penalty_weight'],
        relative_log_ai=relative_log_ai, truth_log_ai=truth_log_ai, truth_reflectivity=truth_reflectivity,
        event_depths_m=event_depths,
    )
    print(f"{well_name}: corr={reconstruction_corr:.3f}, events={event_depths.size}, iterations={solution['iterations']}")

del raw_survey, balanced_survey
metrics_output = pd.DataFrame(metric_rows)
metrics_output.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
display(metrics_output)

## 4. 图件

第一组图看中心井位道：raw、balanced、反射系数、相对 log-AI 增量和重褶积。第二组图看局部剖面，判断反射系数事件是否横向连续，而不是逐道随机出现。

In [ ]:
def section_clip(values, percentile=99.0):
    finite = np.asarray(values)[np.isfinite(values)]
    return max(float(np.percentile(np.abs(finite), percentile)), 1e-9)


for well_name, result in results.items():
    axis = result['axis_m']
    center = result['raw'].shape[0] // 2
    lower = result['zone_top_m'] - 30.0
    upper = result['zone_bottom_m'] + 30.0
    view = (axis >= lower) & (axis <= upper)

    fig, axes = plt.subplots(1, 5, figsize=(14.5, 7.0), constrained_layout=True, sharey=True)
    axes[0].plot(result['raw'][center], axis, color='0.20', linewidth=1.0)
    axes[1].plot(result['balanced'][center], axis, color='#1f77b4', linewidth=1.0)
    axes[2].plot(result['reflectivity'][center], axis, color='#d62728', linewidth=0.9, label='recovered')
    truth_scale = section_clip(result['truth_reflectivity'][view], 95.0)
    recovered_scale = section_clip(result['reflectivity'][center, view], 95.0)
    if np.any(np.isfinite(result['truth_reflectivity'][view])):
        axes[2].plot(
            result['truth_reflectivity'] * recovered_scale / truth_scale, axis, color='0.45',
            linewidth=0.7, alpha=0.75, label='well truth (display-scaled)'
        )
    axes[3].plot(result['relative_log_ai'], axis, color='#2ca02c', linewidth=1.0)
    axes[4].plot(result['raw'][center], axis, color='0.65', linewidth=0.8, label='raw')
    axes[4].plot(result['reconstruction'][center], axis, color='#9467bd', linewidth=1.0, label='reconvolved')
    for event_depth in result['event_depths_m']:
        axes[2].axhline(event_depth, color='#d62728', alpha=0.18, linewidth=0.6)
    for panel in axes:
        panel.axhspan(result['zone_top_m'], result['zone_bottom_m'], color='#f2c6c6', alpha=0.18)
        panel.set_ylim(upper, lower)
    axes[0].set_title('raw seismic')
    axes[1].set_title('balanced seismic')
    axes[2].set_title('sparse reflectivity')
    axes[3].set_title('relative log-AI increment')
    axes[4].set_title('raw / reconvolved')
    axes[0].set_ylabel('TVDSS (m)')
    axes[2].legend(fontsize=7)
    axes[4].legend(fontsize=7)
    fig.suptitle(f'{well_name} | center trace | target event spacing >= {TARGET_MIN_EVENT_SPACING_M:g} m')
    fig.savefig(FIGURE_DIR / f'{well_name}_center_trace.png', bbox_inches='tight')
    plt.close(fig)

    lateral = result['lateral_m']
    extent = [float(lateral[0]), float(lateral[-1]), float(axis[-1]), float(axis[0])]
    raw_clip = section_clip(result['raw'][:, view])
    balanced_clip = section_clip(result['balanced'][:, view])
    reflectivity_clip = section_clip(result['reflectivity'][:, view], 98.0)
    residual_clip = section_clip(result['residual'][:, view])
    fig, axes = plt.subplots(1, 4, figsize=(16.0, 6.5), constrained_layout=True, sharey=True)
    panels = (
        (result['raw'], raw_clip, 'raw seismic'),
        (result['balanced'], balanced_clip, 'balanced seismic'),
        (result['reflectivity'], reflectivity_clip, 'dip-guided sparse reflectivity'),
        (result['residual'], residual_clip, 'raw - reconvolved'),
    )
    for panel, (values, clip, title) in zip(axes, panels):
        panel.imshow(values.T, aspect='auto', cmap='seismic', vmin=-clip, vmax=clip, extent=extent)
        panel.axhline(result['zone_top_m'], color='black', linewidth=0.7)
        panel.axhline(result['zone_bottom_m'], color='black', linewidth=0.7)
        panel.set_ylim(upper, lower)
        panel.set_title(title)
        panel.set_xlabel('xline-relative distance (m)')
    axes[0].set_ylabel('TVDSS (m)')
    fig.suptitle(f'{well_name} | local 1D inversions with dip-aligned multitrace support')
    fig.savefig(FIGURE_DIR / f'{well_name}_section.png', bbox_inches='tight')
    plt.close(fig)

summary = {
    'status': 'complete', 'sample_domain': 'depth', 'sample_unit': 'm', 'depth_basis': 'tvdss',
    'smoke': SMOKE, 'selected_wells': list(SELECTED_WELLS), 'patch_trace_count': 2 * PATCH_RADIUS + 1,
    'target_min_event_spacing_m': TARGET_MIN_EVENT_SPACING_M,
    'raw_seismic': str(RAW_SEISMIC.relative_to(repo_root)),
    'balanced_seismic': str(BALANCED_SEISMIC.relative_to(repo_root)),
    'wavelet': str(WAVELET_FILE.relative_to(repo_root)),
    'interpretation': 'deterministic sparse reflectivity prototype; balanced seismic affects support weights only',
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Figures:', FIGURE_DIR)

## 5. 判读边界

优先查看 `*_section.png` 中的反射系数：弱事件应沿横向连续出现，并且在 residual 中相应减弱。中心道图中的井曲线反射系数只用于检查事件位置，不进入求解。

若增加的事件只存在于单道、不能重褶积回 raw seismic，或主要贴着 patch 边缘出现，则不解释为新增地质体。相对 log-AI 尚未加入真实 LFM，因此只表示反射系数积分后的中高频增量。